In [18]:
# import packages
import cv2
import glob
import numpy as np
import rawpy
import rawpy.enhance
import gc

In [19]:
# Force OpenCV to run entirely on the CPU and completely disable OpenCL acceleration
# addresses an issue I was having (thanks Gemini)
cv2.ocl.setUseOpenCL(False)

In [20]:
# find all sections in the raw data folder. Assumes folder names are structured as "raw_data/<section_name>"
sampleID = sorted(glob.glob("raw_data/*"))

# remove the leading "raw_data/" from each folder name
sampleID = [folder.replace("raw_data/", "") for folder in sampleID]
print(sampleID)

# NOTE: Alternatively, you can mannually define a list of core sections to run as below.
#sampleID = ["ALHIC2502-86"]

['ALHIC2502-102-A', 'ALHIC2502-102-D', 'ALHIC2502-103', 'ALHIC2502-24', 'ALHIC2502-47', 'ALHIC2502-86']


In [ ]:
# set params


# set filepaths
raw_data_dir = "raw_data/"
output_dir = "image_outputs/"

# confidence threshold and scaling - note that you might need to tune these to avoid things blowing up!!!
pano_conf_thresh_lowres = .85
pano_conf_thresh_highres = .95
scale_factor = 0.20  # Downsample to 20% size for feature matching


In [ ]:

# Loop through folders safely
for sample in sampleID:
    print("\n" + "="*50)
    print(f"Running images from: {sample}")
    print("="*50)

    # 1. Locate files dynamically inside the target sample folder
    nef_paths = sorted(glob.glob(f"{raw_data_dir}/{sample}/*.NEF"))
    print(f" Found {len(nef_paths)} NEF files in folder '{sample}'.")
    
    if len(nef_paths) == 0:
        print("  [Warning] No files found. Skipping this folder.")
        continue

    # Step 1: Extract low-res thumbnails to calculate alignment geometry
    print(" Step 1: Extracting low-res thumbnails to calculate alignment...")
    low_res_images = []
    for path in nef_paths:
        with rawpy.imread(path) as raw:
            

            rgb = raw.postprocess(use_camera_wb=True, half_size=True)
        
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        
        # Calculate bounds based on target scale factor
        h, w = bgr.shape[:2]
        target_w = int(w * scale_factor * 2)  # Compensates for fast half_size decode
        target_h = int(h * scale_factor * 2)
        
        small_img = cv2.resize(bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
        low_res_images.append(small_img)

    # Step 2: Test alignment geometry using flat grid SCANS mode (value 1)
    print(" Step 2: Aligning 2D zig-zag grid structure...")
    stitcher = cv2.Stitcher_create(1)
    stitcher.setPanoConfidenceThresh(pano_conf_thresh_lowres)    # set lower confidence threshold - neccisary for some rows
    status, low_res_panorama = stitcher.stitch(low_res_images)

    # Free up memory allocated to thumbnails immediately
    del low_res_images
    gc.collect()

    # check status
    if status != cv2.Stitcher_OK:
        print(f"  [Error] Stitching geometry failed! Error code: {status}")
        print("  Tip: Verify that the frame-to-frame step overlap is at least 20-30%.")
        continue # Skip to next core sample instead of crashing the whole batch script
    print("  Grid layout successfully calculated!")

    # Step 3: High resolution processing with safe RAM footprints
    print(" Step 3: Generating final high-res grid composition...")
    stitcher_high = cv2.Stitcher_create(1)
    stitcher_high.setRegistrationResol(0.6) 
    stitcher_high.setPanoConfidenceThresh(pano_conf_thresh_highres) # <- reduce confidence threshold to avoid missing matches
    high_res_images = []
    for path in nef_paths:
        with rawpy.imread(path) as raw:
            # Full quality decode
            rgb = raw.postprocess(use_camera_wb=True, half_size=False)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        high_res_images.append(bgr)
    print("  Assembling canvas matrices...")
    status, high_res_panorama = stitcher_high.stitch(high_res_images)

    # Instantly wipe the high-res list elements out of memory cache
    del high_res_images
    gc.collect()

    # check status, save file
    if status == cv2.Stitcher_OK:
        # Standardized file pathing structure
        output_path = f"{output_dir}/{sample}_stitched_output.jpg"
        cv2.imwrite(output_path, high_res_panorama)
        print(f"  Success! Stitched grid saved to: {output_path}")
        
        # Completely drop panorama matrix representation to reset memory baseline
        del high_res_panorama
        gc.collect()
    else:
        print(f"  [Error] Highmultip resolution layout composition processing failed: {status}")


Running images from: ALHIC2502-102-A
 Found 52 NEF files in folder 'ALHIC2502-102-A'.
 Step 1: Extracting low-res thumbnails to calculate alignment...
  [Error] Lens correction failed for raw_data//ALHIC2502-102-A/ALHIC_2502_102A_01.NEF: module 'rawpy.enhance' has no attribute 'correct_lens'
  [Error] Lens correction failed for raw_data//ALHIC2502-102-A/ALHIC_2502_102A_02.NEF: module 'rawpy.enhance' has no attribute 'correct_lens'
  [Error] Lens correction failed for raw_data//ALHIC2502-102-A/ALHIC_2502_102A_03.NEF: module 'rawpy.enhance' has no attribute 'correct_lens'
  [Error] Lens correction failed for raw_data//ALHIC2502-102-A/ALHIC_2502_102A_04.NEF: module 'rawpy.enhance' has no attribute 'correct_lens'
  [Error] Lens correction failed for raw_data//ALHIC2502-102-A/ALHIC_2502_102A_05.NEF: module 'rawpy.enhance' has no attribute 'correct_lens'
  [Error] Lens correction failed for raw_data//ALHIC2502-102-A/ALHIC_2502_102A_06.NEF: module 'rawpy.enhance' has no attribute 'correct_le